# Experiment 6 — Automatic-path Matched Trad vs Frozen Lexical

**Run in a NEW notebook; do not overwrite your baseline or final-model notebooks.**
This version handles files split between Windows Desktop/FYP/QuishingDataset
and your Jupyter working folder (the behaviour seen in your earlier code).
The five files do NOT need to be put together.

Inputs:
- qr_codes_29.pickle, qr_codes_29_labels.pickle (often Desktop\FYP\QuishingDataset)
- qr_full_feature_dataset_v2.csv (often Jupyter current folder)
- FINAL_10feature_XGB_for_deployment.pkl (often Jupyter current folder)
- phase14B_final_robust_features.json (often Jupyter current folder)

The experiment reuses the original_indices, labels, hostname grouping and
the existing frozen lexical model, then trains Trad-style raw-pixel XGBoost
only on the 7,936 development samples. Evaluation uses exactly 1,777 holdout
samples. Output appears in a new folder INSIDE the Jupyter working directory.

Important: only load .pickle/.pkl files from sources you trust.

## 1. Configuration: automatic search first, optional manual override
You should normally run this cell WITHOUT changing anything.
If a file is not found, paste its exact path into MANUAL_FILE_PATHS below.

In [ ]:
from pathlib import Path
import json
import pickle
import time
import os
import warnings
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
)

SEED = 42
N_BOOTSTRAPS = 500
# Optional absolute or relative paths for any missing inputs.
# Example: 'qr_codes_29.pickle': r'C:\Users\LENOVO\Desktop\FYP\QuishingDataset\qr_codes_29.pickle',
MANUAL_FILE_PATHS = {
    'qr_codes_29.pickle': None,
    'qr_codes_29_labels.pickle': None,
    'qr_full_feature_dataset_v2.csv': None,
    'FINAL_10feature_XGB_for_deployment.pkl': None,
    'phase14B_final_robust_features.json': None,
}

# If your Jupyter folder lives in a different drive or an unusual directory,
# you can list its folder path below. Paths are searched individually.
ADDITIONAL_SEARCH_FOLDERS = [
    # r'D:\MyJupyterFolder',
]

# These locations were found in YOUR previous notebooks:
OLD_NOTEBOOK_QR_DATA_DIR = r'C:\Users\LENOVO\Desktop\FYP\QuishingDataset'
OLD_NOTEBOOK_ALT_DIR = r'C:\Users\LENOVO\FYP\QuishingDataset'
OLD_NOTEBOOK_ARCHIVE_DIR = r'C:\Users\LENOVO\Desktop\FYP\archive'

REQUIRED = list(MANUAL_FILE_PATHS)
CWD = Path.cwd().resolve()
OUTPUT_DIR = CWD / 'experiment6_matched_trad_vs_lexical'

## 2. Find the files wherever your old code saved them
Jupyter saves relative paths inside `Path.cwd()`; QR pickles can be on Desktop.
The selected path for every file is printed below. No data is silently rebuilt.

In [ ]:
def _unique_roots(paths):
    result = []
    seen = set()
    for p in paths:
        if p is None: continue
        try:
            pp = Path(p).expanduser().resolve()
            k = str(pp).lower()
            if pp.is_dir() and k not in seen:
                result.append(pp); seen.add(k)
        except (OSError, ValueError):
            continue
    return result

home = Path.home()
script_dir = Path(__file__).resolve().parent if '__file__' in globals() else None
roots = _unique_roots([
    CWD, script_dir,
    CWD.parent, CWD.parent / 'QuishingDataset',
    CWD / 'QuishingDataset', CWD / 'FYP', CWD / 'archive',
    *ADDITIONAL_SEARCH_FOLDERS,
    OLD_NOTEBOOK_QR_DATA_DIR,
    OLD_NOTEBOOK_ALT_DIR,
    OLD_NOTEBOOK_ARCHIVE_DIR,
    home / 'Desktop' / 'FYP' / 'QuishingDataset',
    home / 'Desktop' / 'FYP',
    home / 'Desktop' / 'FYP' / 'archive',
    home / 'OneDrive' / 'Desktop' / 'FYP' / 'QuishingDataset',
    home / 'OneDrive' / 'Desktop' / 'FYP',
    home / 'FYP' / 'QuishingDataset',
    home / 'Documents', home / 'Downloads',
    home / 'Jupyter', home / 'Jupyter Notebook',
    home / 'Desktop',
])

# Avoid slow/irrelevant virtual environments and system folders.
SKIP_DIRS = {
    '.git', '.ipynb_checkpoints', '__pycache__', 'node_modules',
    '.venv', 'venv', 'env', 'AppData', 'Library', 'site-packages',
    'anaconda3', 'miniconda3', 'Anaconda3',
    'experiment6_matched_trad_vs_lexical',
}

def candidates_for(filename):
    aliases = [filename]
    # In a previous upload the feature CSV was accidentally named *.xls
    if filename == 'qr_full_feature_dataset_v2.csv':
        aliases.append('qr_full_feature_dataset_v2.xls')
    found = []
    seen = set()
    def add(path):
        path = Path(path)
        try:
            if path.is_file():
                key = str(path.resolve()).lower()
                if key not in seen:
                    found.append(path.resolve()); seen.add(key)
        except OSError:
            pass
    # First: direct children of current / known project folders.
    for root in roots:
        for name in aliases: add(root / name)
    # Then search nested Jupyter project folders (at most 3 levels).
    for root in roots:
        for folder, dirs, names in os.walk(root):
            depth = len(Path(folder).relative_to(root).parts)
            dirs[:] = sorted(d for d in dirs if d not in SKIP_DIRS and not d.startswith('.')) if depth < 3 else []
            for name in aliases:
                if name in names: add(Path(folder) / name)
    return found

FILES = {}
print('Current Jupyter/Python directory:', CWD)
print('Existing search roots:', len(roots))
for filename in REQUIRED:
    override = MANUAL_FILE_PATHS[filename]
    if override:
        p = Path(override).expanduser()
        if not p.is_file():
            raise FileNotFoundError(f'Manual path for {filename} is invalid: {p}')
        chosen = p.resolve()
        all_found = [chosen]
    else:
        all_found = candidates_for(filename)
        if not all_found:
            raise FileNotFoundError(
                f'Cannot find {filename}. Current directory = {CWD}.\n'
                f'Paths examined include:\n' + '\n'.join(str(r) for r in roots) +
                f'\n\nPlease put the actual path in MANUAL_FILE_PATHS[{filename!r}]'
            )
        # Prefer exact .csv over an accidentally renamed .xls if both found.
        csv_matches = [x for x in all_found if x.suffix.lower() == '.csv']
        chosen = csv_matches[0] if csv_matches else all_found[0]
        if len(all_found) > 1:
            print(f'  [NOTE] {filename}: {len(all_found)} copies found; choosing first priority.')
            for alt in all_found[:6]: print('     candidate:', alt)
    FILES[filename] = chosen
    print(f'  [FOUND] {filename}\n          {chosen}')

# Support previous upload's .xls extension only if it is in fact CSV content.
feature_path = FILES['qr_full_feature_dataset_v2.csv']
if feature_path.suffix.lower() == '.xls':
    with open(feature_path, 'rb') as fh:
        head = fh.read(512)
    if b'original_index' not in head and b'normalized_url' not in head:
        raise ValueError(f'{feature_path} is a genuine XLS, not a CSV renamed to XLS. Export it to CSV.')

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
print('Output folder:', OUTPUT_DIR)

## 3. Load data and validate original-index / label correspondence
No QR decoding, no QR regeneration, no re-ordering of the cleaned CSV.

In [ ]:
with open(FILES['qr_codes_29.pickle'], 'rb') as fh:
    qr_codes = np.asarray(pickle.load(fh))
with open(FILES['qr_codes_29_labels.pickle'], 'rb') as fh:
    raw_labels = np.asarray(pickle.load(fh)).astype(int)

full_df = pd.read_csv(FILES['qr_full_feature_dataset_v2.csv'])
with open(FILES['phase14B_final_robust_features.json'], encoding='utf-8') as fh:
    config = json.load(fh)
feature_names = list(config['features'])
lexical_model = joblib.load(FILES['FINAL_10feature_XGB_for_deployment.pkl'])

assert qr_codes.shape == (9987, 69, 69), qr_codes.shape
assert raw_labels.shape == (9987,), raw_labels.shape
assert len(full_df) == 9713, len(full_df)
assert full_df['original_index'].notna().all()
assert full_df['original_index'].is_unique, 'original_index must be unique'
assert full_df['hostname'].notna().all()
assert set(full_df['label'].unique()).issubset({0,1})
assert len(feature_names) == 10 and len(set(feature_names)) == 10
assert all(f in full_df for f in feature_names)
assert not full_df[feature_names].isna().any().any(), 'Missing lexical features'
assert np.isin(qr_codes, [0,1]).all(), 'QR values are expected to be 0/1'

orig = full_df['original_index'].astype(int).to_numpy()
labels = full_df['label'].astype(int).to_numpy()
hostnames = full_df['hostname'].astype(str).to_numpy()
assert (orig >= 0).all() and (orig < len(qr_codes)).all()
assert np.array_equal(raw_labels[orig], labels), 'QR / CSV label alignment failed'

model_features = list(getattr(lexical_model, 'feature_names_in_', feature_names))
assert model_features == feature_names, (
    'Frozen model expects a different feature order!\n'
    f'Model={model_features}\nConfig={feature_names}'
)
print('Original QR images:', qr_codes.shape)
print('Cleaned aligned pairs:', len(full_df))
print('Features:', feature_names)
print('Labels:', dict(zip(*np.unique(labels, return_counts=True))))
print('QR-to-CSV index and label alignment: PASS')

## 3. Re-create the exact frozen-model holdout
Use the **same rows, same order, same random seed, same hostname grouping** as
the clean lexical notebook. The test set remains untouched during raw-pixel training.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
dev_idx, holdout_idx = next(splitter.split(full_df[feature_names], labels, groups=hostnames))
train_hosts = set(hostnames[dev_idx]); test_hosts = set(hostnames[holdout_idx])
assert not train_hosts.intersection(test_hosts), 'Hostname leakage!'
assert (len(dev_idx), len(holdout_idx)) == (7936, 1777), (
    len(dev_idx), len(holdout_idx)
)
assert len(np.unique(labels[holdout_idx])) == 2
print(f'Development: {len(dev_idx)} | Holdout: {len(holdout_idx)}')
print('Hostname overlap:', len(train_hosts & test_hosts))
print('Holdout phishing share:', round(labels[holdout_idx].mean(), 4))

## 4. Re-evaluate the **existing**, frozen lexical detector
This should reproduce the paper's F1=0.8811, AUC=0.9509, MCC=0.7641.

In [ ]:
def metrics_of(y_true, scores):
    predictions = (np.asarray(scores) >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0,1]).ravel()
    return {
        'accuracy': accuracy_score(y_true, predictions),
        'precision': precision_score(y_true, predictions, zero_division=0),
        'recall': recall_score(y_true, predictions, zero_division=0),
        'f1': f1_score(y_true, predictions, zero_division=0),
        'roc_auc': roc_auc_score(y_true, scores),
        'mcc': matthews_corrcoef(y_true, predictions),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }

y_holdout = labels[holdout_idx]
X_lex_holdout = full_df.iloc[holdout_idx][feature_names]
lexical_scores = lexical_model.predict_proba(X_lex_holdout)[:,1]
lexical_metrics = metrics_of(y_holdout, lexical_scores)
print('Frozen lexical holdout:', lexical_metrics)
assert abs(lexical_metrics['f1'] - 0.881087) < 0.001, 'Frozen lexical F1 differs from validated run'
assert abs(lexical_metrics['roc_auc'] - 0.950949) < 0.001, 'Frozen lexical AUC differs from validated run'
assert abs(lexical_metrics['mcc'] - 0.764140) < 0.001, 'Frozen lexical MCC differs from validated run'

## 5. Train the Trad-style raw-pixel reference model (only on development data)
Exactly 69*69=4,761 raw module values are used per QR code. This mirrors the
reproduction notebook's original XGBoost choice: learning_rate=0.2,
n_estimators=150, threshold=0.5. This is **Trad-style**, not guaranteed to match
Trad & Chehab's precise original training split or hyperparameter search.

In [ ]:
raw_train = qr_codes[orig[dev_idx]].reshape(len(dev_idx), 69 * 69).astype(np.float32)
raw_holdout = qr_codes[orig[holdout_idx]].reshape(len(holdout_idx), 69 * 69).astype(np.float32)
assert raw_train.shape == (7936, 4761)
assert raw_holdout.shape == (1777, 4761)

raw_model = XGBClassifier(
    learning_rate=0.2,
    n_estimators=150,
    eval_metric='logloss',
    random_state=SEED,
    n_jobs=4,
)
start = time.perf_counter()
raw_model.fit(raw_train, labels[dev_idx])
training_seconds = time.perf_counter() - start
raw_scores = raw_model.predict_proba(raw_holdout)[:,1]
raw_metrics = metrics_of(y_holdout, raw_scores)
print(f'Trad-style raw-pixel model trained in {training_seconds:.1f} s')
print('Raw-pixel holdout:', raw_metrics)

## 6. Matched performance table and paired hostname-cluster bootstrap
Difference = Lexical minus Raw-pixel, on exactly the same holdout labels.
Resampling the hostnames (not just individual rows) accounts for URLs sharing
the same hostname. No hyperparameters are selected using the holdout.

In [ ]:
summary_df = pd.DataFrame([
    {'model': 'Trad-style raw-pixel XGBoost', 'input_dim': 4761, 'n': len(y_holdout), **raw_metrics},
    {'model': 'Frozen 10-feature lexical XGBoost', 'input_dim': 10, 'n': len(y_holdout), **lexical_metrics},
])
print('\n=== Matched holdout comparison ===')
print(summary_df[['model','n','roc_auc','f1','mcc','accuracy','precision','recall']].round(5).to_string(index=False))

host_holdout = hostnames[holdout_idx]
unique_hosts, inverse = np.unique(host_holdout, return_inverse=True)
host_row_indices = [np.flatnonzero(inverse == k) for k in range(len(unique_hosts))]
rng = np.random.default_rng(SEED)
boot_rows = []
for b in range(N_BOOTSTRAPS):
    sampled_groups = rng.integers(0, len(unique_hosts), size=len(unique_hosts))
    chosen_rows = np.concatenate([host_row_indices[g] for g in sampled_groups])
    yy = y_holdout[chosen_rows]
    if np.unique(yy).size < 2:
        continue
    rr = metrics_of(yy, raw_scores[chosen_rows])
    ll = metrics_of(yy, lexical_scores[chosen_rows])
    boot_rows.append({
        'bootstrap': b,
        'delta_auc_lex_minus_raw': ll['roc_auc'] - rr['roc_auc'],
        'delta_f1_lex_minus_raw': ll['f1'] - rr['f1'],
        'delta_mcc_lex_minus_raw': ll['mcc'] - rr['mcc'],
    })
bootstrap_df = pd.DataFrame(boot_rows)
assert len(bootstrap_df) >= int(N_BOOTSTRAPS * 0.95), 'Too many invalid bootstrap resamples'
paired_rows = []
for metric, col in [('roc_auc','delta_auc_lex_minus_raw'),
                    ('f1','delta_f1_lex_minus_raw'),
                    ('mcc','delta_mcc_lex_minus_raw')]:
    observed = lexical_metrics[metric] - raw_metrics[metric]
    ci_low, ci_high = np.percentile(bootstrap_df[col], [2.5,97.5])
    paired_rows.append({'metric':metric,'delta_lex_minus_raw':observed,
                        'ci95_lower':ci_low,'ci95_upper':ci_high,
                        'cluster':'hostname','bootstrap_replicates':len(bootstrap_df)})
paired_ci_df = pd.DataFrame(paired_rows)
print('\n=== Paired hostname bootstrap (lexical minus raw pixel) ===')
print(paired_ci_df.round(5).to_string(index=False))

## 7. Save inspectable results and figure
Keep this experiment separate from the published manuscript until reviewed.

In [ ]:
from sklearn import __version__ as sklearn_version
from xgboost import __version__ as xgboost_version

holdout_predictions = full_df.iloc[holdout_idx][['original_index','url','hostname','label']].copy()
holdout_predictions['raw_pixel_probability'] = raw_scores
holdout_predictions['lexical_probability'] = lexical_scores
holdout_predictions['raw_pixel_prediction'] = (raw_scores >= 0.5).astype(int)
holdout_predictions['lexical_prediction'] = (lexical_scores >= 0.5).astype(int)
holdout_predictions['raw_pixel_correct'] = (holdout_predictions['raw_pixel_prediction'] == y_holdout)
holdout_predictions['lexical_correct'] = (holdout_predictions['lexical_prediction'] == y_holdout)

summary_df.to_csv(OUTPUT_DIR / 'matched_holdout_summary.csv', index=False)
paired_ci_df.to_csv(OUTPUT_DIR / 'paired_hostname_bootstrap_ci.csv', index=False)
bootstrap_df.to_csv(OUTPUT_DIR / 'bootstrap_replicates.csv', index=False)
holdout_predictions.to_csv(OUTPUT_DIR / 'matched_holdout_predictions.csv', index=False)
joblib.dump(raw_model, OUTPUT_DIR / 'trad_style_raw_pixel_xgb_grouped.pkl')
with open(OUTPUT_DIR / 'run_metadata.json', 'w', encoding='utf-8') as f:
    json.dump({
        'seed':SEED, 'bootstrap_seed':SEED, 'n_bootstraps':N_BOOTSTRAPS,
        'development_n':len(dev_idx), 'holdout_n':len(holdout_idx),
        'development_host_count':len(train_hosts), 'holdout_host_count':len(test_hosts),
        'hostname_overlap':len(train_hosts & test_hosts),
        'raw_pixel_features':4761, 'lexical_features':feature_names,
        'raw_model_params':raw_model.get_params(),
        'training_seconds':training_seconds,
        'sklearn_version':sklearn_version, 'xgboost_version':xgboost_version,
        'source_files': {k: str(v) for k,v in FILES.items()},
        'interpretation':'Matched holdout comparison; not exact Trad published numerical replication.',
    }, f, indent=2, default=str)

fig, ax = plt.subplots(figsize=(7.8,4.2))
metric_names = ['roc_auc','f1','mcc']
x = np.arange(len(metric_names))
bar_width = 0.36
ax.bar(x-bar_width/2,[raw_metrics[k] for k in metric_names], bar_width,label='Trad-style raw pixels')
ax.bar(x+bar_width/2,[lexical_metrics[k] for k in metric_names], bar_width,label='Frozen lexical (10)')
ax.set_xticks(x, ['ROC–AUC','F1','MCC'])
ax.set_ylim(0,1.05)
ax.set_ylabel('Score (held-out data)')
ax.set_title('Matched hostname-disjoint holdout (N=1,777)')
ax.legend(loc='lower right')
ax.grid(axis='y',alpha=.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'matched_comparison.png',dpi=180,bbox_inches='tight')
plt.show()
print('\nSaved results to:', OUTPUT_DIR.resolve())
print('NEXT: Discuss whether the matched difference supports your claim; do not compare with Trad publication random split as if it were the same protocol.')